Plotting for Salmon align, using bam files from STAR

Load packages

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.decomposition import PCA

PCA

Load datasets created in R

In [2]:
norm_counts = pd.read_csv("C:/Users/Sebas/OneDrive/Dokument/Master courses/MASTER THESIS/R project-Master Thesis/salmon_alignment_normalized_counts_transcript_consistent_new_filtering.csv", index_col=0)
norm_counts

,ERR12356072,ERR12383247,ERR12383248,ERR12383249,ERR12383250,ERR12383251,ERR12383252,ERR12383253,ERR12383254,ERR12383255,...,ERR12383308,ERR12383309,ERR12383310,ERR12383311,ERR12383312,ERR12383313,ERR12383314,ERR12383315,ERR12383316,ERR12383317
g2.t1,8.002841,8.151411,8.209028,9.086502,8.645975,9.408128,7.882847,8.148355,8.204822,8.897636,...,8.511710,8.258551,8.578235,7.791385,7.921404,8.343431,7.944615,8.397098,8.461890,9.003019
g3.t1,7.257903,7.095574,7.544380,7.267473,7.973541,7.583449,7.043668,7.483398,7.257510,7.506517,...,6.683246,7.057310,6.580052,6.339533,6.489995,6.569957,6.584527,7.506409,7.913673,7.520098
g4.t1,7.727833,7.561088,7.484671,7.232027,7.136546,7.083352,7.563366,7.463401,7.761316,7.173026,...,7.434388,7.435921,7.361691,7.661990,7.611379,7.496180,7.487826,7.163412,7.291544,6.739584
g6.t1,8.330465,8.685266,8.247890,7.849397,8.112580,7.966472,8.677889,8.303212,8.449781,8.056227,...,7.875153,7.966807,8.275250,8.409870,8.639480,8.593760,8.612008,7.958489,7.797175,7.629655
g7.t1,6.712268,6.893115,6.888505,6.374720,6.487940,6.426381,7.090857,7.106738,6.915581,6.206652,...,6.934025,6.366742,6.285332,6.979603,6.748595,6.803883,6.980717,6.782472,6.786697,6.604117
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
g34843.t1,6.081501,6.263195,6.126713,6.331512,5.801107,6.397025,6.181731,6.067702,6.250281,6.555267,...,6.031119,5.743406,5.948749,6.214908,5.633162,5.686667,6.061698,6.220392,6.318326,6.242201
g34884.t1,5.428180,5.621512,5.428180,5.428180,5.428180,5.984724,5.616986,5.428180,5.428180,5.428180,...,6.133512,5.958729,5.620774,5.645884,5.572160,5.798858,5.428180,5.846876,5.653472,5.993211
g34922.t1,5.797458,5.882880,5.865492,5.844682,5.805155,6.246387,5.868663,5.428180,5.713396,5.428180,...,5.858338,5.837263,5.877552,5.971981,5.639897,5.822561,5.771453,6.036554,5.428180,5.811321
g35167.t1,5.668280,5.826385,6.168327,6.198141,5.428180,6.232163,5.843006,5.909243,6.168433,5.853447,...,5.979363,5.690034,5.873075,5.867790,5.757698,5.729339,5.769402,6.270373,6.199435,6.481765


In [3]:
#Metadata
metadata = pd.read_csv("C:/Users/Sebas/OneDrive/Dokument/Master courses/MASTER THESIS/R project-Master Thesis/dominance_meta_corrected_outlier_corrected.csv")

#make sure order matches norm_counts
metadata = metadata.set_index('Run')
metadata = metadata.reindex(norm_counts.columns)
# Now Run is the index, and metadata is aligned with counts
print("Metadata index (samples):", metadata.index[:5].tolist())
print("Counts columns:", norm_counts.columns[:5].tolist())
print("Match?", all(metadata.index == norm_counts.columns))
metadata

Metadata index (samples): ['ERR12356072', 'ERR12383247', 'ERR12383248', 'ERR12383249', 'ERR12383250']
Counts columns: ['ERR12356072', 'ERR12383247', 'ERR12383248', 'ERR12383249', 'ERR12383250']
Match? True


,Bases,BioProject,BioSample,Experiment,sample_name,TF ID,Sample ID,Cross,Family,Sex,original_fastq_name_R1,original_fastq_name_R2,Notes
ERR12356072,21027960416,PRJEB70958,SAMEA114860228,ERX11733017,Sample 1 males genotype BA heterozygote,TF2581-10-e3,10e 3,13:20 (M) + 42:13 (F),"2,3,4,",F,TF-2581-10-e3_S67_L001_R1_001.fastq.gz,TF-2581-10-e3_S67_L001_R2_001.fastq.gz,NaN
ERR12383247,14794943760,PRJEB70958,SAMEA114860213,ERX11759665,Sample 1 females genotype AA homozygote,TF2581-11,11,13:20 (M) + 42:13 (F),"2,3,4,",F,TF-2581-11_S9_L001_R1_001.fastq.gz,TF-2581-11_S9_L001_R2_001.fastq.gz,NaN
ERR12383248,15610248328,PRJEB70958,SAMEA114860214,ERX11759666,Sample 2 females genotype AA homozygote,TF2581-12,12,13:20 (M) + 42:13 (F),"2,3,4,",F,TF-2581-12_S10_L001_R1_001.fastq.gz,TF-2581-12_S10_L001_R2_001.fastq.gz,NaN
ERR12383249,9412089720,PRJEB70958,SAMEA114860215,ERX11759667,Sample 3 females genotype AA homozygote,TF2581-13,13,42:13 (M) + 13:20 (F),"1,4,8",M,TF-2581-13_S11_L001_R1_001.fastq.gz,TF-2581-13_S11_L001_R2_001.fastq.gz,NaN
ERR12383250,9959631122,PRJEB70958,SAMEA114860216,ERX11759668,Sample 1 males genotype AA homozygote,TF2581-14-e2,14e 2,42:13 (M) + 13:20 (F),"1,4,8",M,TF-2581-14-e2_S68_L001_R1_001.fastq.gz,TF-2581-14-e2_S68_L001_R2_001.fastq.gz,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ERR12383313,27406244206,PRJEB70958,SAMEA114860279,ERX11759731,Sample 1 females genotype FF homozygote,TF2581-71,71,4:18 + 4:18,"2,6,9",F,TF-2581-71_S59_L001_R1_001.fastq.gz,TF-2581-71_S59_L001_R2_001.fastq.gz,"Had TF ID: TF2581-70, and Sample ID: 71"
ERR12383314,13458064958,PRJEB70958,SAMEA114860280,ERX11759732,Sample 2 females genotype FF homozygote,TF2581-72,72,4:18 + 4:18,"2,6,9",F,TF-2581-72_S60_L001_R1_001.fastq.gz,TF-2581-72_S60_L001_R2_001.fastq.gz,"Had TF ID: TF2581-71, and Sample ID: 72"
ERR12383315,11890141660,PRJEB70958,SAMEA114860281,ERX11759733,Sample 3 females genotype FF homozygote,TF2581-7,7,13:20 (M) + 42:13 (F),"2,3,4,",M,TF-2581-7_S7_L001_R1_001.fastq.gz,TF-2581-7_S7_L001_R2_001.fastq.gz,NaN
ERR12383316,13692674262,PRJEB70958,SAMEA114860282,ERX11759734,Sample 1 males genotype FF homozygote,TF2581-8,8,13:20 (M) + 42:13 (F),"2,3,4,",M,TF-2581-8_S8_L001_R1_001.fastq.gz,TF-2581-8_S8_L001_R2_001.fastq.gz,NaN


In [4]:
# create transpose
vst_t = norm_counts.T

pca = PCA(n_components=2)
pca_scores = pca.fit_transform(vst_t)

pca_df = metadata.copy()
pca_df['PC1'] = pca_scores[:,0]
pca_df['PC2'] = pca_scores[:,1]

# Explained variance
pc1_var = pca.explained_variance_ratio_[0] * 100
pc2_var = pca.explained_variance_ratio_[1] * 100

# Define your own color mapping
color_map = {'M': 'blue', 'F': 'red'}  

fig = px.scatter(
    pca_df,
    x='PC1',
    y='PC2',
    color='Sex',  
    color_discrete_map=color_map,          
    hover_name=pca_df.index,
    title=f"Salmon-Align: Male vs. Female PCA of VST-normalized counts"
)

fig.update_layout(
    xaxis_title=f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)",
    yaxis_title=f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)",
)
fig.show()


Volcano PLot

In [5]:
results_full_annot = pd.read_csv("C:/Users/Sebas/OneDrive/Dokument/Master courses/MASTER THESIS/R project-Master Thesis/salmon_align_dominance_DE_sex_results_new_filtering.csv", float_precision='legacy')
results_full_annot

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,start,...,HOG,OG,Gene.Tree.Parent.Clade,PFAMs,GOs,EC,KEGG_ko,KEGG_Pathway,COG_category,eggNOG_OGs
0,237.825076,0.820721,0.128795,6.372287,1.862300e-10,4.530465e-10,g2.t1,g2,utg000001l,220384.0,...,N0.HOG0012111,OG0011460,n0,COX8,-,-,ko:K02273,"ko00190,ko01100,ko04260,ko04714,ko04932,ko0501...",I,"2FBM0@1|root,2TCUK@2759|Eukaryota,398E5@33154|..."
1,73.844964,0.663084,0.181367,3.656039,2.561425e-04,4.432381e-04,g3.t1,g3,utg000001l,227675.0,...,N0.HOG0004052,OG0003481,n0,"PX,Pkinase,Pkinase_Tyr,WH2","GO:0003674,GO:0005488,GO:0005515,GO:0005543,GO...",3.1.26.5,"ko:K14529,ko:K17543","ko03013,map03013",DUZ,"KOG2101@1|root,KOG2101@2759|Eukaryota,396Y9@33..."
2,115.316375,-0.604007,0.168972,-3.574591,3.507763e-04,6.013374e-04,g4.t1,g4,utg000001l,245866.0,...,N0.HOG0004826,OG0004221,n0,"Porphobil_deam,Porphobil_deamC","GO:0001101,GO:0001666,GO:0003674,GO:0003824,GO...",2.5.1.61,ko:K01749,"ko00860,ko01100,ko01110,ko01120,map00860,map01...",H,"COG0181@1|root,KOG2892@2759|Eukaryota,38D6W@33..."
3,251.432475,-0.890110,0.062282,-14.291522,2.471491e-46,1.808786e-45,g6.t1,g6,utg000001l,263441.0,...,N0.HOG0015574,OG0014921,n0,THAP,-,-,-,-,B,"2E6V3@1|root,2SDHR@2759|Eukaryota"
4,42.078202,-0.956817,0.112738,-8.487085,2.118852e-17,6.771986e-17,g7.t1,g7,utg000001l,371755.0,...,N0.HOG0010167,OG0009526,n0,"CH,LRR_8","GO:0002682,GO:0002683,GO:0002685,GO:0002686,GO...",-,-,-,Z,"COG4886@1|root,KOG0532@2759|Eukaryota,38HCP@33..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16558,11.656909,0.290288,0.220053,1.319173,1.871114e-01,2.293982e-01,g34843.t1,g34843,utg003648l,5656.0,...,N0.HOG0012740,OG0012089,n0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16559,1.364347,2.303175,0.590766,3.898621,9.674188e-05,1.722282e-04,g34884.t1,g34884,utg003700l,19710.0,...,N0.HOG0000536,OG0000347,n0,Retrotrans_gag,-,-,-,-,S,"2D3G7@1|root,2SRFN@2759|Eukaryota,3AMW9@33154|..."
16560,2.364626,0.412563,0.360914,1.143107,2.529942e-01,3.014236e-01,g34922.t1,g34922,utg003714l,1.0,...,N0.HOG0000406,OG0000247,n0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16561,3.988477,1.153179,0.359851,3.204606,1.352475e-03,2.209006e-03,g35167.t1,g35167,utg003885l,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Replace the zeros in padj with 1e-308 avoid log10 issues
results_full_annot["padj_safe"] = results_full_annot["padj"].replace(0, 1e-308).fillna(1)

# Add the negative log 10 padj for plotting
results_full_annot["neglog10_padj"] = -np.log10(results_full_annot["padj_safe"])


# Add significance to differentially expressed transcripts 
results_full_annot["significant"] = (
    (results_full_annot["padj"] < 0.05) &
    (results_full_annot["log2FoldChange"].abs() > 1)
)

# Add a label to the significant transcripts
results_full_annot["label"] = results_full_annot["transcript_id"].where(results_full_annot["significant"], "")

results_full_annot

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,start,...,GOs,EC,KEGG_ko,KEGG_Pathway,COG_category,eggNOG_OGs,padj_safe,neglog10_padj,significant,label
0,237.825076,0.820721,0.128795,6.372287,1.862300e-10,4.530465e-10,g2.t1,g2,utg000001l,220384.0,...,-,-,ko:K02273,"ko00190,ko01100,ko04260,ko04714,ko04932,ko0501...",I,"2FBM0@1|root,2TCUK@2759|Eukaryota,398E5@33154|...",4.530465e-10,9.343857,False,
1,73.844964,0.663084,0.181367,3.656039,2.561425e-04,4.432381e-04,g3.t1,g3,utg000001l,227675.0,...,"GO:0003674,GO:0005488,GO:0005515,GO:0005543,GO...",3.1.26.5,"ko:K14529,ko:K17543","ko03013,map03013",DUZ,"KOG2101@1|root,KOG2101@2759|Eukaryota,396Y9@33...",4.432381e-04,3.353363,False,
2,115.316375,-0.604007,0.168972,-3.574591,3.507763e-04,6.013374e-04,g4.t1,g4,utg000001l,245866.0,...,"GO:0001101,GO:0001666,GO:0003674,GO:0003824,GO...",2.5.1.61,ko:K01749,"ko00860,ko01100,ko01110,ko01120,map00860,map01...",H,"COG0181@1|root,KOG2892@2759|Eukaryota,38D6W@33...",6.013374e-04,3.220882,False,
3,251.432475,-0.890110,0.062282,-14.291522,2.471491e-46,1.808786e-45,g6.t1,g6,utg000001l,263441.0,...,-,-,-,-,B,"2E6V3@1|root,2SDHR@2759|Eukaryota",1.808786e-45,44.742613,False,
4,42.078202,-0.956817,0.112738,-8.487085,2.118852e-17,6.771986e-17,g7.t1,g7,utg000001l,371755.0,...,"GO:0002682,GO:0002683,GO:0002685,GO:0002686,GO...",-,-,-,Z,"COG4886@1|root,KOG0532@2759|Eukaryota,38HCP@33...",6.771986e-17,16.169284,False,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16558,11.656909,0.290288,0.220053,1.319173,1.871114e-01,2.293982e-01,g34843.t1,g34843,utg003648l,5656.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2.293982e-01,0.639410,False,
16559,1.364347,2.303175,0.590766,3.898621,9.674188e-05,1.722282e-04,g34884.t1,g34884,utg003700l,19710.0,...,-,-,-,-,S,"2D3G7@1|root,2SRFN@2759|Eukaryota,3AMW9@33154|...",1.722282e-04,3.763896,True,g34884.t1
16560,2.364626,0.412563,0.360914,1.143107,2.529942e-01,3.014236e-01,g34922.t1,g34922,utg003714l,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,3.014236e-01,0.520823,False,
16561,3.988477,1.153179,0.359851,3.204606,1.352475e-03,2.209006e-03,g35167.t1,g35167,utg003885l,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2.209006e-03,2.655803,True,g35167.t1


In [7]:
fig = px.scatter(
    results_full_annot,
    x="log2FoldChange",
    y="neglog10_padj",
    hover_name="transcript_id",
    hover_data=["padj", "HOG", "PFAMs"],  # Use the columns that already exist
    color="significant",
    color_discrete_map={True: "red", False: "blue"},
    title="Salmon-Align: Volcano Plot (Male vs. Female Transcript Expression)"
)

fig.update_traces(textposition='top center', textfont_size=8)

fig.update_layout(
    xaxis_title="log2 Fold Change (male vs female)",
    yaxis_title="-log10(padj (FDR))",
)

# Horizontal FDR=0.05 cutoff
padj_cut = -np.log10(0.05)

fig.add_hline(
    y=padj_cut,
    line_dash="dash",
    line_color="grey",
    annotation_text="padj = 0.05",
    annotation_position="bottom right"
)

# Vertical log2FC cutoffs
fig.add_vline(
    x=-1,
    line_dash="dash",
    line_color="grey",
    annotation_text="log2FC = -1",
    annotation_position="top left"
)
fig.add_vline(
    x=1,
    line_dash="dash",
    line_color="grey",
    annotation_text="log2FC = 1",
    annotation_position="top right"
)

fig.show()


#DE genes
sig_counts = results_full_annot["significant"].sum()
higher_in_m = ((results_full_annot["significant"]) & 
               (results_full_annot["log2FoldChange"] > 0)).sum()
higher_in_f = ((results_full_annot["significant"]) & 
               (results_full_annot["log2FoldChange"] < 0)).sum()

print(f"Total significant transcripts: {sig_counts}")
print(f"Higher in males: {higher_in_m}")
print(f"Higher in females: {higher_in_f}")

Total significant transcripts: 6309
Higher in males: 4184
Higher in females: 2125
